[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/optimization/08_stochastic_optimization_for_ml/first_principles.ipynb)

# Topic 08: Stochastic Optimization for Machine Learning

## 1. First-Principles Intuition & Motivation

### 1.1 The Finite Sum and the Cost of an Exact Gradient

Supervised learning minimizes an **empirical risk** that is an average over $N$ examples,

$$
f(\mathbf{x}) = \frac{1}{N}\sum_{i=1}^N f_i(\mathbf{x}), \qquad f_i(\mathbf{x}) = \ell\left(\text{model}_{\mathbf{x}}(\mathbf{a}_i),\, b_i\right)
$$

standing in for the **expected risk** $F(\mathbf{x}) = \mathbb{E}_{(\mathbf{a},b)\sim\mathcal{D}}\left[\ell(\cdot)\right]$
over the unknown data distribution. Exact gradient descent needs

$$
\nabla f(\mathbf{x}) = \frac{1}{N}\sum_{i=1}^N \nabla f_i(\mathbf{x})
$$

which costs one full pass over the dataset *per step*. With $N = 10^9$ that is absurd: the gradient is an
average, and averages are exactly the quantity that random sampling estimates well. A single example
already points roughly downhill; a batch of $32$ points downhill with quantifiable accuracy.

The bargain is therefore explicit: **trade gradient accuracy for the number of updates**. In the time one
exact gradient step takes, stochastic gradient descent takes $N/B$ noisy steps. The entire theory of this
module studies whether that trade is a good one, and under what step-size discipline the noise does not
destroy convergence.

### 1.2 Noise Is a Tax, Not a Wall

Write the stochastic gradient as its mean plus a fluctuation:

$$
\mathbf{g}_k = \nabla f(\mathbf{x}_k) + \boldsymbol{\epsilon}_k, \qquad \mathbb{E}\left[\boldsymbol{\epsilon}_k \mid \mathbf{x}_k\right] = \mathbf{0}, \qquad \mathbb{E}\left[\lVert \boldsymbol{\epsilon}_k\rVert^2 \mid \mathbf{x}_k\right] \le \frac{\sigma^2}{B}
$$

Two facts organize everything that follows:

- **Unbiasedness** means the *drift* of the iterate is still the true descent direction, so in expectation
  the algorithm makes deterministic progress.
- **Variance** means each step also injects energy. Feeding $\mathbf{x}_{k+1} = \mathbf{x}_k - \alpha\mathbf{g}_k$
  into the descent lemma produces the master inequality

$$
\mathbb{E}\left[f(\mathbf{x}_{k+1})\right] \le f(\mathbf{x}_k) - \alpha\left(1 - \frac{L\alpha}{2}\right)\lVert \nabla f(\mathbf{x}_k)\rVert^2 + \frac{L\alpha^2\sigma^2}{2B}
$$

The first correction is the familiar deterministic descent term; the second is the **noise tax**. Note the
asymmetry in $\alpha$: progress is $O(\alpha)$ while the tax is $O(\alpha^2)$. Small steps therefore make
the tax negligible — but also make progress slow. That single trade-off dictates every step-size schedule
in machine learning.

### 1.3 The Noise Ball and the Two Regimes

Because the tax does not vanish as $\lVert \nabla f\rVert \to 0$, a *constant* step size cannot converge to
$\mathbf{x}^*$. Instead the iterates equilibrate where progress balances tax:

$$
\alpha\mu\left(f - f^*\right) \sim \frac{L\alpha^2\sigma^2}{2B} \quad\Longrightarrow\quad f - f^* \sim \frac{L\alpha\sigma^2}{2\mu B}
$$

a **noise ball** of radius proportional to $\alpha\sigma^2/B$. Training therefore has two visibly distinct
phases:

1. **Transient phase.** The gradient dominates the noise; the loss falls at essentially the deterministic
   rate — geometrically for strongly convex objectives.
2. **Stationary phase.** The iterate diffuses inside the noise ball; the loss curve flattens and rattles.
   Progress resumes only by *shrinking the ball*: lowering $\alpha$ (the learning-rate decay every
   practitioner has seen produce a step down in the loss curve), enlarging $B$, or reducing $\sigma$ by a
   control variate.

Robbins and Monro (1951) turned this into a schedule: let $\alpha_k \to 0$ slowly enough to keep travelling
($\sum_k\alpha_k = \infty$) but fast enough to quiet the noise ($\sum_k\alpha_k^2 \lt \infty$). Infinite
total fuel, finite total shaking.

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition 1 (Finite-sum and expected risk).** For $f_i:\mathbb{R}^n\to\mathbb{R}$,

$$
f(\mathbf{x}) = \frac{1}{N}\sum_{i=1}^N f_i(\mathbf{x}), \qquad F(\mathbf{x}) = \mathbb{E}_{\xi}\left[f_\xi(\mathbf{x})\right]
$$

**Definition 2 (Stochastic gradient oracle).** A random vector $\mathbf{g}(\mathbf{x},\xi)$ is an
**unbiased** stochastic gradient if

$$
\mathbb{E}_{\xi}\left[\mathbf{g}(\mathbf{x},\xi)\right] = \nabla f(\mathbf{x}) \qquad \text{for every } \mathbf{x}
$$

and has **variance bounded by $\sigma^2$** if
$\mathbb{E}_\xi\lVert \mathbf{g}(\mathbf{x},\xi)-\nabla f(\mathbf{x})\rVert^2 \le \sigma^2$.

**Definition 3 (Mini-batch estimator).** For an index multiset $\mathcal{B}$ of size $B$ drawn uniformly
with replacement from $\{1,\dots,N\}$,

$$
\mathbf{g}_{\mathcal{B}}(\mathbf{x}) = \frac{1}{B}\sum_{i\in\mathcal{B}}\nabla f_i(\mathbf{x})
$$

**Definition 4 (SGD).** Given step sizes $\alpha_k \gt 0$,

$$
\mathbf{x}_{k+1} = \mathbf{x}_k - \alpha_k\,\mathbf{g}_k, \qquad \mathbb{E}\left[\mathbf{g}_k\mid\mathcal{F}_k\right] = \nabla f(\mathbf{x}_k)
$$

where $\mathcal{F}_k$ is the $\sigma$-algebra generated by everything up to iteration $k$.

**Definition 5 (Robbins-Monro conditions).** A step-size schedule $\{\alpha_k\}$ is admissible if

$$
\alpha_k \gt 0, \qquad \sum_{k=0}^{\infty}\alpha_k = \infty, \qquad \sum_{k=0}^{\infty}\alpha_k^2 \lt \infty
$$

Canonical example: $\alpha_k = \alpha_0 / (k+1)^p$ with $p \in (1/2,\ 1]$.

**Definition 6 (SVRG estimator).** With a *snapshot* $\tilde{\mathbf{x}}$ at which the full gradient
$\nabla f(\tilde{\mathbf{x}})$ has been computed, and $i$ drawn uniformly,

$$
\mathbf{g}^{\mathrm{SVRG}}(\mathbf{x}) = \nabla f_i(\mathbf{x}) - \nabla f_i(\tilde{\mathbf{x}}) + \nabla f(\tilde{\mathbf{x}})
$$

**Definition 7 (SGD with heavy-ball momentum).**

$$
\mathbf{m}_{k+1} = \beta\mathbf{m}_k + \mathbf{g}_k, \qquad \mathbf{x}_{k+1} = \mathbf{x}_k - \alpha\,\mathbf{m}_{k+1}
$$

**Definition 8 (AdaGrad and Adam).** AdaGrad accumulates squared gradients per coordinate,

$$
v_{k,j} = \sum_{t\le k}g_{t,j}^2, \qquad x_{k+1,j} = x_{k,j} - \frac{\alpha}{\sqrt{v_{k,j}}+\varepsilon}\,g_{k,j}
$$

Adam replaces the sums by exponential moving averages with bias correction,

$$
\mathbf{m}_k = \beta_1\mathbf{m}_{k-1} + (1-\beta_1)\mathbf{g}_k, \quad \mathbf{v}_k = \beta_2\mathbf{v}_{k-1} + (1-\beta_2)\mathbf{g}_k^{\odot2}
$$

$$
\hat{\mathbf{m}}_k = \frac{\mathbf{m}_k}{1-\beta_1^k}, \quad \hat{\mathbf{v}}_k = \frac{\mathbf{v}_k}{1-\beta_2^k}, \qquad \mathbf{x}_{k+1} = \mathbf{x}_k - \alpha\,\frac{\hat{\mathbf{m}}_k}{\sqrt{\hat{\mathbf{v}}_k}+\varepsilon}
$$

**Theorem 1 (Unbiasedness and the $1/B$ variance law).** If each $\nabla f_i$ has
$\frac{1}{N}\sum_i\lVert \nabla f_i(\mathbf{x})-\nabla f(\mathbf{x})\rVert^2 = \sigma^2(\mathbf{x})$, then
for sampling with replacement,

$$
\mathbb{E}\left[\mathbf{g}_{\mathcal{B}}(\mathbf{x})\right] = \nabla f(\mathbf{x}), \qquad \mathbb{E}\left\lVert \mathbf{g}_{\mathcal{B}}(\mathbf{x})-\nabla f(\mathbf{x})\right\rVert^2 = \frac{\sigma^2(\mathbf{x})}{B}
$$

For sampling *without* replacement the variance carries the finite-population correction factor
$\frac{N-B}{N-1}$, which vanishes at $B = N$.

**Theorem 2 (Expected descent inequality).** If $f$ is $L$-smooth and $\mathbf{g}_k$ is unbiased with
variance at most $\sigma_B^2 = \sigma^2/B$, then for constant $\alpha \gt 0$,

$$
\mathbb{E}\left[f(\mathbf{x}_{k+1})\mid\mathcal{F}_k\right] \le f(\mathbf{x}_k) - \alpha\left(1-\frac{L\alpha}{2}\right)\lVert \nabla f(\mathbf{x}_k)\rVert^2 + \frac{L\alpha^2\sigma_B^2}{2}
$$

**Theorem 3 (Constant step: linear convergence to a noise ball).** If in addition $f$ is $\mu$-strongly
convex and $\alpha \le 1/L$, then

$$
\mathbb{E}\left[f(\mathbf{x}_k)\right] - f^* \le (1-\alpha\mu)^k\left(f(\mathbf{x}_0)-f^*\right) + \frac{L\alpha\sigma_B^2}{2\mu}
$$

The first term decays geometrically; the second is the noise floor, proportional to $\alpha$ and to $1/B$.

**Theorem 4 (Decreasing steps: $O(1/k)$ for strongly convex objectives).** If $f$ is $\mu$-strongly
convex, $\mathbb{E}\lVert \mathbf{g}_k\rVert^2 \le G^2$, and $\alpha_k = \frac{1}{\mu k}$, then

$$
\mathbb{E}\left\lVert \mathbf{x}_k - \mathbf{x}^*\right\rVert^2 \le \frac{G^2}{\mu^2 k}
$$

and consequently $\mathbb{E}[f(\mathbf{x}_k)] - f^* \le \frac{LG^2}{2\mu^2k} = O(1/k)$. This rate is
*minimax optimal* for stochastic first-order methods on strongly convex problems: no algorithm using only
noisy gradients can beat $\Omega(1/k)$.

**Theorem 5 (Nonconvex stationarity rate).** For $L$-smooth $f$ bounded below, SGD with
$\alpha = \min\left\{\frac{1}{L},\ \frac{c}{\sigma_B\sqrt{K}}\right\}$ satisfies

$$
\frac{1}{K}\sum_{k=0}^{K-1}\mathbb{E}\left\lVert \nabla f(\mathbf{x}_k)\right\rVert^2 = O\!\left(\frac{1}{\sqrt{K}}\right)
$$

so $O(\epsilon^{-4})$ stochastic gradient evaluations suffice to reach
$\mathbb{E}\lVert \nabla f\rVert \le \epsilon$ — versus $O(\epsilon^{-2})$ for exact gradients.

**Theorem 6 (SVRG variance reduction).** For $L$-smooth convex $f_i$, the SVRG estimator is unbiased and

$$
\mathbb{E}\left\lVert \mathbf{g}^{\mathrm{SVRG}}(\mathbf{x})-\nabla f(\mathbf{x})\right\rVert^2 \le 4L\left[\left(f(\mathbf{x})-f^*\right)+\left(f(\tilde{\mathbf{x}})-f^*\right)\right]
$$

so the variance vanishes as both $\mathbf{x}$ and $\tilde{\mathbf{x}}$ approach the optimum. Consequently
SVRG attains a **linear** rate on strongly convex finite sums with $O\left((N+\kappa)\log(1/\epsilon)\right)$
gradient evaluations.

**Theorem 7 (Adam bias correction).** If the gradient second moments are stationary,
$\mathbb{E}[\mathbf{g}_t^{\odot2}] = \mathbb{E}[\mathbf{g}^{\odot2}]$ for all $t$, and
$\mathbf{v}_0 = \mathbf{0}$, then

$$
\mathbb{E}\left[\mathbf{v}_k\right] = \left(1-\beta_2^{\,k}\right)\mathbb{E}\left[\mathbf{g}^{\odot2}\right]
$$

so $\hat{\mathbf{v}}_k = \mathbf{v}_k/(1-\beta_2^k)$ is unbiased; the identical computation applies to
$\mathbf{m}_k$ with $\beta_1$.

**Theorem 8 (Polyak-Juditsky averaging).** Under $\alpha_k = \alpha_0k^{-p}$ with $p\in(1/2,1)$, the
average $\bar{\mathbf{x}}_K = \frac{1}{K}\sum_{k \lt K}\mathbf{x}_k$ satisfies

$$
\sqrt{K}\left(\bar{\mathbf{x}}_K - \mathbf{x}^*\right) \xrightarrow{d} \mathcal{N}\!\left(\mathbf{0},\ H^{-1}\Sigma H^{-1}\right), \qquad H = \nabla^2f(\mathbf{x}^*)
$$

attaining the asymptotic covariance of the maximum-likelihood estimator — statistically optimal, and
achieved without knowing $H$.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 1: Mini-Batch Unbiasedness and the $1/B$ Variance Law

**Claim.** With $\mathcal{B} = \{i_1,\dots,i_B\}$ drawn i.i.d. uniformly from $\{1,\dots,N\}$,
$\mathbb{E}[\mathbf{g}_{\mathcal{B}}] = \nabla f$ and
$\mathbb{E}\lVert \mathbf{g}_{\mathcal{B}}-\nabla f\rVert^2 = \sigma^2/B$.

**Step 1 (unbiasedness of a single draw).** For $i$ uniform on $\{1,\dots,N\}$,

$$
\mathbb{E}\left[\nabla f_i(\mathbf{x})\right] = \sum_{j=1}^N \frac{1}{N}\nabla f_j(\mathbf{x}) = \nabla f(\mathbf{x})
$$

This is nothing but the statement that the finite sum *is* an expectation under the uniform distribution.

**Step 2 (unbiasedness of the batch).** Linearity of expectation:

$$
\mathbb{E}\left[\mathbf{g}_{\mathcal{B}}\right] = \frac{1}{B}\sum_{r=1}^{B}\mathbb{E}\left[\nabla f_{i_r}\right] = \frac{1}{B}\cdot B\,\nabla f(\mathbf{x}) = \nabla f(\mathbf{x})
$$

**Step 3 (variance of a single draw).** Define $\boldsymbol{\epsilon}_i = \nabla f_i(\mathbf{x})-\nabla f(\mathbf{x})$,
so $\mathbb{E}[\boldsymbol{\epsilon}_i] = \mathbf{0}$ and

$$
\mathbb{E}\lVert \boldsymbol{\epsilon}_i\rVert^2 = \frac{1}{N}\sum_{j=1}^N\left\lVert \nabla f_j(\mathbf{x})-\nabla f(\mathbf{x})\right\rVert^2 = \sigma^2(\mathbf{x})
$$

**Step 4 (independence kills the cross terms).**

$$
\mathbb{E}\left\lVert \mathbf{g}_{\mathcal{B}}-\nabla f\right\rVert^2 = \frac{1}{B^2}\,\mathbb{E}\left\lVert \sum_{r=1}^B\boldsymbol{\epsilon}_{i_r}\right\rVert^2 = \frac{1}{B^2}\left[\sum_{r}\mathbb{E}\lVert \boldsymbol{\epsilon}_{i_r}\rVert^2 + \sum_{r\neq s}\underbrace{\mathbb{E}\left[\boldsymbol{\epsilon}_{i_r}^T\boldsymbol{\epsilon}_{i_s}\right]}_{= \mathbb{E}[\boldsymbol{\epsilon}_{i_r}]^T\mathbb{E}[\boldsymbol{\epsilon}_{i_s}] = 0}\right]
$$

$$
= \frac{B\sigma^2}{B^2} = \frac{\sigma^2}{B}
$$

**Step 5 (without replacement).** If $\mathcal{B}$ is a uniform random subset of size $B$, the draws are
negatively correlated and the classical finite-population correction applies:

$$
\mathbb{E}\left\lVert \mathbf{g}_{\mathcal{B}}-\nabla f\right\rVert^2 = \frac{\sigma^2}{B}\cdot\frac{N-B}{N-1}
$$

which correctly gives $0$ at $B = N$ (the full gradient is exact). $\blacksquare$

**Reading the law.** Variance falls like $1/B$ but the *standard deviation* only like $1/\sqrt{B}$, while
compute grows like $B$. Accuracy per unit compute is therefore constant — the fact behind every
batch-size argument in Section 4.

### Proof 2: The Expected Descent Inequality

**Claim.** For $L$-smooth $f$ and unbiased $\mathbf{g}_k$ with conditional variance at most $\sigma_B^2$,

$$
\mathbb{E}\left[f(\mathbf{x}_{k+1})\mid\mathcal{F}_k\right] \le f(\mathbf{x}_k) - \alpha\left(1-\frac{L\alpha}{2}\right)\lVert \nabla f(\mathbf{x}_k)\rVert^2 + \frac{L\alpha^2\sigma_B^2}{2}
$$

**Step 1 (descent lemma).** $L$-smoothness gives, for any $\mathbf{y}$,

$$
f(\mathbf{y}) \le f(\mathbf{x}_k) + \nabla f(\mathbf{x}_k)^T(\mathbf{y}-\mathbf{x}_k) + \frac{L}{2}\lVert \mathbf{y}-\mathbf{x}_k\rVert^2
$$

Apply it at $\mathbf{y} = \mathbf{x}_{k+1} = \mathbf{x}_k - \alpha\mathbf{g}_k$:

$$
f(\mathbf{x}_{k+1}) \le f(\mathbf{x}_k) - \alpha\,\nabla f(\mathbf{x}_k)^T\mathbf{g}_k + \frac{L\alpha^2}{2}\lVert \mathbf{g}_k\rVert^2
$$

**Step 2 (take the conditional expectation).** Using $\mathbb{E}[\mathbf{g}_k\mid\mathcal{F}_k] = \nabla f(\mathbf{x}_k)$
for the linear term:

$$
\mathbb{E}\left[\nabla f(\mathbf{x}_k)^T\mathbf{g}_k\mid\mathcal{F}_k\right] = \lVert \nabla f(\mathbf{x}_k)\rVert^2
$$

**Step 3 (the second moment splits).** The bias-variance decomposition for a random vector gives

$$
\mathbb{E}\left[\lVert \mathbf{g}_k\rVert^2\mid\mathcal{F}_k\right] = \left\lVert \mathbb{E}[\mathbf{g}_k\mid\mathcal{F}_k]\right\rVert^2 + \mathbb{E}\left[\lVert \mathbf{g}_k-\nabla f(\mathbf{x}_k)\rVert^2\mid\mathcal{F}_k\right] \le \lVert \nabla f(\mathbf{x}_k)\rVert^2 + \sigma_B^2
$$

**Step 4 (assemble).**

$$
\mathbb{E}\left[f(\mathbf{x}_{k+1})\mid\mathcal{F}_k\right] \le f(\mathbf{x}_k) - \alpha\lVert \nabla f_k\rVert^2 + \frac{L\alpha^2}{2}\left(\lVert \nabla f_k\rVert^2 + \sigma_B^2\right)
$$

Collecting the $\lVert \nabla f_k\rVert^2$ terms yields the claim. $\blacksquare$

**Two readings.**

- Setting $\sigma_B = 0$ recovers the deterministic descent inequality, with the familiar requirement
  $\alpha \lt 2/L$ for the coefficient $\alpha(1-L\alpha/2)$ to be positive.
- With $\sigma_B \gt 0$, no choice of $\alpha$ makes the right side smaller than
  $f(\mathbf{x}_k) + \frac{L\alpha^2\sigma_B^2}{2}$ once $\nabla f(\mathbf{x}_k) = \mathbf{0}$: the
  algorithm keeps moving at the optimum. Hence the noise ball is unavoidable at fixed $\alpha$.

### Proof 3: Constant Step Size Converges Linearly to a Noise Ball

**Claim.** For $L$-smooth, $\mu$-strongly convex $f$ and $0 \lt \alpha \le 1/L$,

$$
\mathbb{E}\left[f(\mathbf{x}_k)\right]-f^* \le (1-\alpha\mu)^k\left(f(\mathbf{x}_0)-f^*\right) + \frac{L\alpha\sigma_B^2}{2\mu}
$$

**Step 1 (simplify the descent inequality).** With $\alpha \le 1/L$ we have $1 - \frac{L\alpha}{2} \ge \frac12$,
so Proof 2 gives

$$
\mathbb{E}\left[f(\mathbf{x}_{k+1})\mid\mathcal{F}_k\right] \le f(\mathbf{x}_k) - \frac{\alpha}{2}\lVert \nabla f(\mathbf{x}_k)\rVert^2 + \frac{L\alpha^2\sigma_B^2}{2}
$$

**Step 2 (apply the PL / strong-convexity inequality).** Strong convexity implies gradient dominance,

$$
\lVert \nabla f(\mathbf{x})\rVert^2 \ge 2\mu\left(f(\mathbf{x})-f^*\right)
$$

(minimize both sides of the strong convexity lower bound over $\mathbf{y}$). Substituting,

$$
\mathbb{E}\left[f(\mathbf{x}_{k+1})\mid\mathcal{F}_k\right] - f^* \le \left(1-\alpha\mu\right)\left(f(\mathbf{x}_k)-f^*\right) + \frac{L\alpha^2\sigma_B^2}{2}
$$

**Step 3 (unroll the affine recursion).** Write $\delta_k = \mathbb{E}[f(\mathbf{x}_k)]-f^*$,
$\rho = 1-\alpha\mu \in [0,1)$ and $c = \frac{L\alpha^2\sigma_B^2}{2}$. Taking total expectations,
$\delta_{k+1} \le \rho\,\delta_k + c$, hence

$$
\delta_k \le \rho^k\delta_0 + c\sum_{j=0}^{k-1}\rho^{j} \le \rho^k\delta_0 + \frac{c}{1-\rho} = (1-\alpha\mu)^k\delta_0 + \frac{L\alpha^2\sigma_B^2}{2\alpha\mu}
$$

which is the claim after simplifying $\frac{L\alpha^2\sigma_B^2}{2\alpha\mu} = \frac{L\alpha\sigma_B^2}{2\mu}$.
$\blacksquare$

**Step 4 (the fixed point is the noise ball).** The recursion's fixed point
$\delta_\infty = \frac{c}{1-\rho} = \frac{L\alpha\sigma^2}{2\mu B}$ is exactly the balance predicted in
Section 1.3. Three levers shrink it, each with a different cost:

$$
\delta_\infty \propto \frac{\alpha\,\sigma^2}{\mu\,B}: \quad \text{halve } \alpha \ (\text{slower transient}), \quad \text{double } B \ (\text{2}\times\text{ compute/step}), \quad \text{reduce } \sigma \ (\text{SVRG})
$$

**Step 5 (a schedule from the bound).** Halving $\alpha$ each time the loss plateaus makes $\delta$ track a
staircase: each drop in $\alpha$ halves the floor and the run resumes its geometric transient. This is the
theoretical justification for step-decay learning-rate schedules.

### Proof 4: Robbins-Monro Step Sizes Give the $O(1/k)$ Rate

**Claim.** For $\mu$-strongly convex $f$ with $\mathbb{E}\lVert \mathbf{g}_k\rVert^2 \le G^2$ and
$\alpha_k = \frac{1}{\mu k}$,

$$
a_k := \mathbb{E}\lVert \mathbf{x}_k - \mathbf{x}^*\rVert^2 \le \frac{G^2}{\mu^2 k}
$$

**Step 1 (expand the squared distance).**

$$
\lVert \mathbf{x}_{k+1}-\mathbf{x}^*\rVert^2 = \lVert \mathbf{x}_k-\mathbf{x}^*\rVert^2 - 2\alpha_k\,\mathbf{g}_k^T\left(\mathbf{x}_k-\mathbf{x}^*\right) + \alpha_k^2\lVert \mathbf{g}_k\rVert^2
$$

**Step 2 (condition and use unbiasedness).**

$$
\mathbb{E}\left[\mathbf{g}_k^T(\mathbf{x}_k-\mathbf{x}^*)\mid\mathcal{F}_k\right] = \nabla f(\mathbf{x}_k)^T\left(\mathbf{x}_k-\mathbf{x}^*\right)
$$

**Step 3 (strong monotonicity).** Strong convexity gives, with $\nabla f(\mathbf{x}^*) = \mathbf{0}$,

$$
\nabla f(\mathbf{x}_k)^T\left(\mathbf{x}_k-\mathbf{x}^*\right) = \left(\nabla f(\mathbf{x}_k)-\nabla f(\mathbf{x}^*)\right)^T\left(\mathbf{x}_k-\mathbf{x}^*\right) \ge \mu\lVert \mathbf{x}_k-\mathbf{x}^*\rVert^2
$$

**Step 4 (the recursion).** Combining and taking total expectations,

$$
a_{k+1} \le \left(1 - 2\mu\alpha_k\right)a_k + \alpha_k^2G^2
$$

**Step 5 (induction with $\alpha_k = 1/(\mu k)$).** Let $Q = G^2/\mu^2$; we claim $a_k \le Q/k$. The base
case $k=1$ follows from $a_1 \le \alpha_0^2G^2$ with a suitable initialization convention. Assume
$a_k \le Q/k$. Then $2\mu\alpha_k = 2/k$ and $\alpha_k^2G^2 = Q/k^2$, so

$$
a_{k+1} \le \left(1-\frac{2}{k}\right)\frac{Q}{k} + \frac{Q}{k^2} = Q\,\frac{k-2}{k^2} + \frac{Q}{k^2} = Q\,\frac{k-1}{k^2}
$$

Finally $\frac{k-1}{k^2} \le \frac{1}{k+1}$ because $(k-1)(k+1) = k^2-1 \le k^2$. Hence
$a_{k+1} \le \frac{Q}{k+1}$, completing the induction. $\blacksquare$

**Step 6 (why both Robbins-Monro conditions are needed).**

- $\sum_k\alpha_k = \infty$: the total distance the drift term can cover is $O(\sum_k\alpha_k)$; a summable
  schedule (e.g. $\alpha_k = 2^{-k}$) can leave the iterate stranded far from $\mathbf{x}^*$ forever.
- $\sum_k\alpha_k^2 \lt \infty$: the accumulated injected noise energy is $O(\sigma^2\sum_k\alpha_k^2)$; if
  it diverges the iterate never settles. The harmonic schedule $\alpha_k \propto 1/k$ is precisely on the
  boundary of both conditions, and $\alpha_k \propto k^{-p}$ works for $p\in(1/2,1]$.

### Proof 5: SVRG — Unbiasedness and Vanishing Variance

**Claim.** $\mathbf{g}^{\mathrm{SVRG}}(\mathbf{x}) = \nabla f_i(\mathbf{x}) - \nabla f_i(\tilde{\mathbf{x}}) + \nabla f(\tilde{\mathbf{x}})$
is unbiased, and for $L$-smooth convex $f_i$,

$$
\mathbb{E}\left\lVert \mathbf{g}^{\mathrm{SVRG}}(\mathbf{x})-\nabla f(\mathbf{x})\right\rVert^2 \le 4L\left[\left(f(\mathbf{x})-f^*\right)+\left(f(\tilde{\mathbf{x}})-f^*\right)\right]
$$

**Step 1 (unbiasedness).** With $i$ uniform and $\tilde{\mathbf{x}}$ fixed,

$$
\mathbb{E}\left[\mathbf{g}^{\mathrm{SVRG}}\right] = \underbrace{\mathbb{E}\left[\nabla f_i(\mathbf{x})\right]}_{=\nabla f(\mathbf{x})} - \underbrace{\mathbb{E}\left[\nabla f_i(\tilde{\mathbf{x}})\right]}_{=\nabla f(\tilde{\mathbf{x}})} + \nabla f(\tilde{\mathbf{x}}) = \nabla f(\mathbf{x})
$$

The last two terms are a **control variate**: a random quantity with known mean, subtracted to cancel
fluctuation without moving the mean.

**Step 2 (variance is a second moment about the mean).** Using
$\mathbb{E}\lVert \mathbf{z}-\mathbb{E}\mathbf{z}\rVert^2 \le \mathbb{E}\lVert \mathbf{z}\rVert^2$ applied
to $\mathbf{z} = \nabla f_i(\mathbf{x})-\nabla f_i(\tilde{\mathbf{x}})$ (whose mean is
$\nabla f(\mathbf{x})-\nabla f(\tilde{\mathbf{x}})$),

$$
\mathbb{E}\left\lVert \mathbf{g}^{\mathrm{SVRG}}-\nabla f(\mathbf{x})\right\rVert^2 \le \mathbb{E}\left\lVert \nabla f_i(\mathbf{x})-\nabla f_i(\tilde{\mathbf{x}})\right\rVert^2
$$

**Step 3 (split around the optimum).** With $(\mathbf{a}+\mathbf{b})$-splitting and
$\lVert \mathbf{u}+\mathbf{v}\rVert^2 \le 2\lVert \mathbf{u}\rVert^2 + 2\lVert \mathbf{v}\rVert^2$,

$$
\mathbb{E}\left\lVert \nabla f_i(\mathbf{x})-\nabla f_i(\tilde{\mathbf{x}})\right\rVert^2 \le 2\,\mathbb{E}\left\lVert \nabla f_i(\mathbf{x})-\nabla f_i(\mathbf{x}^*)\right\rVert^2 + 2\,\mathbb{E}\left\lVert \nabla f_i(\tilde{\mathbf{x}})-\nabla f_i(\mathbf{x}^*)\right\rVert^2
$$

**Step 4 (smoothness + convexity co-coercivity).** For $L$-smooth convex $h$ with minimum $h^*$,

$$
\lVert \nabla h(\mathbf{x})-\nabla h(\mathbf{y})\rVert^2 \le 2L\left[h(\mathbf{x})-h(\mathbf{y})-\nabla h(\mathbf{y})^T(\mathbf{x}-\mathbf{y})\right]
$$

Applying this to each $f_i$ at $\mathbf{y}=\mathbf{x}^*$ and averaging over $i$ (the cross terms
$\frac1N\sum_i\nabla f_i(\mathbf{x}^*) = \nabla f(\mathbf{x}^*) = \mathbf{0}$ vanish) gives

$$
\mathbb{E}\left\lVert \nabla f_i(\mathbf{x})-\nabla f_i(\mathbf{x}^*)\right\rVert^2 \le 2L\left(f(\mathbf{x})-f^*\right)
$$

**Step 5 (combine).** Substituting into Step 3 yields the claimed bound. $\blacksquare$

**Step 6 (why this gives a linear rate).** As $\mathbf{x}, \tilde{\mathbf{x}} \to \mathbf{x}^*$ both terms
$\to 0$, so the effective $\sigma_B^2$ in Proof 3 **shrinks with the suboptimality** rather than remaining
a fixed floor. The noise ball collapses as the iterate approaches the optimum, and SVRG converges
geometrically with total cost

$$
O\!\left(\left(N + \kappa\right)\log\frac{1}{\epsilon}\right) \quad \text{versus} \quad O\!\left(\frac{\kappa\sigma^2}{\mu\epsilon}\right) \ \text{for plain SGD}
$$

The price is one full-gradient snapshot every epoch — worthwhile for moderate $N$ and high accuracy, but
rarely used in deep learning where $N$ is enormous and $\epsilon$ is loose.

### Proof 6: Momentum and Adam — Effective Step Size and Bias Correction

**Part A: momentum rescales the step by $1/(1-\beta)$.**

Unroll Definition 7 with $\mathbf{m}_0 = \mathbf{0}$:

$$
\mathbf{m}_{k} = \sum_{t=1}^{k}\beta^{\,k-t}\mathbf{g}_t, \qquad \mathbf{x}_{k+1} = \mathbf{x}_k - \alpha\sum_{t=1}^{k+1}\beta^{\,k+1-t}\mathbf{g}_t
$$

If the gradients were constant, $\mathbf{g}_t \equiv \mathbf{g}$, then
$\mathbf{m}_k \to \frac{\mathbf{g}}{1-\beta}$, so the **steady-state step** is

$$
\alpha_{\mathrm{eff}} = \frac{\alpha}{1-\beta}
$$

Raising $\beta$ from $0.9$ to $0.99$ multiplies the effective step by $10$ — the classic cause of
"momentum made my training explode".

**Variance reduction from averaging.** If the $\mathbf{g}_t$ are unbiased with variance $\sigma_B^2$ and
(idealized) independent, then

$$
\operatorname{Var}\left[(1-\beta)\mathbf{m}_k\right] = (1-\beta)^2\sum_{t}\beta^{2(k-t)}\sigma_B^2 \xrightarrow[k\to\infty]{} \frac{(1-\beta)^2}{1-\beta^2}\sigma_B^2 = \frac{1-\beta}{1+\beta}\,\sigma_B^2
$$

so the *normalized* momentum buffer behaves like an average over an effective window of
$\frac{1+\beta}{1-\beta}$ samples ($19$ samples at $\beta=0.9$). Momentum is thus simultaneously an
accelerator (Topic 03) and a variance reducer.

**Part B: Adam's bias correction, derived.**

Unroll $\mathbf{v}_k = \beta_2\mathbf{v}_{k-1}+(1-\beta_2)\mathbf{g}_k^{\odot2}$ from $\mathbf{v}_0 = \mathbf{0}$:

$$
\mathbf{v}_k = (1-\beta_2)\sum_{t=1}^{k}\beta_2^{\,k-t}\,\mathbf{g}_t^{\odot2}
$$

Take expectations, assuming stationarity $\mathbb{E}[\mathbf{g}_t^{\odot2}] = \mathbb{E}[\mathbf{g}^{\odot2}]$:

$$
\mathbb{E}\left[\mathbf{v}_k\right] = (1-\beta_2)\sum_{t=1}^{k}\beta_2^{\,k-t}\;\mathbb{E}\left[\mathbf{g}^{\odot2}\right] = (1-\beta_2)\cdot\frac{1-\beta_2^{\,k}}{1-\beta_2}\;\mathbb{E}\left[\mathbf{g}^{\odot2}\right] = \left(1-\beta_2^{\,k}\right)\mathbb{E}\left[\mathbf{g}^{\odot2}\right]
$$

The estimator is biased **low** by exactly the factor $1-\beta_2^k$, purely because of the artificial zero
initialization. Dividing it out gives the unbiased $\hat{\mathbf{v}}_k$; the same algebra with $\beta_1$
gives $\hat{\mathbf{m}}_k$. $\blacksquare$

**Magnitude of the correction.** With $\beta_2 = 0.999$: at $k=1$ the factor is $10^{-3}$ (a $1000\times$
correction), at $k = 100$ it is $0.095$, at $k=1000$ it is $0.632$. Without correction the first steps
would be enormous (dividing by a near-zero $\sqrt{\mathbf{v}}$) — which is also why Adam is often paired
with a warmup schedule for the residual instability.

**Adam as a preconditioner.** The update $\alpha\hat{\mathbf{m}}/(\sqrt{\hat{\mathbf{v}}}+\varepsilon)$ is
a diagonal preconditioner $P \approx \operatorname{diag}(\sqrt{\mathbb{E}[g_j^2]})$. On a quadratic with
Hessian $\operatorname{diag}(\lambda_j)$ the raw gradient scales like $\lambda_j$ and the denominator like
$\lambda_j$, so the effective per-coordinate step is nearly *uniform* — this is precisely why Adam is
insensitive to the wildly different gradient scales across layers of a deep network.

## 4. Computational & Algorithmic Insights

### 4.1 Batch Size Is a Compute-Allocation Decision

Per Proof 1, the variance is $\sigma^2/B$ while the cost per step is $\Theta(B)$. Fix a budget of $C$
gradient evaluations:

$$
\text{steps} = \frac{C}{B}, \qquad \text{noise floor} \propto \frac{\alpha\sigma^2}{B}
$$

In the *noise-dominated* regime the two effects cancel exactly: doubling $B$ halves the floor but also
halves the number of updates, so the loss reached after budget $C$ is unchanged — unless $\alpha$ is
increased too. That is the theoretical content of the **linear scaling rule**: when $B \to cB$, set
$\alpha \to c\alpha$ to keep $\alpha/B$ (hence the noise floor) fixed while restoring the total distance
travelled $\sum\alpha_k$.

The rule breaks in two places:

- **Curvature ceiling.** Stability still requires $\alpha \lt 2/L$; once $c\alpha$ crosses that, larger
  batches cannot be compensated. This defines a **critical batch size** $B_{\mathrm{crit}}$ beyond which
  extra samples buy almost no wall-clock progress (McCandlish et al.'s gradient-noise-scale analysis).
- **Warmup.** Immediately after initialization the local $L$ is large and the linear-scaling $\alpha$ is
  unstable, so large-batch runs ramp $\alpha$ linearly over the first few epochs — the Goyal et al.
  "1 hour ImageNet" recipe.

Practically: measure the gradient noise scale $\mathcal{B}_{\text{noise}} = \frac{\operatorname{tr}\Sigma}{\lVert \nabla f\rVert^2}$;
batches well below it are compute-efficient, batches well above it are merely time-efficient.

### 4.2 Schedules, Averaging and Diagnostics

**Reading a loss curve through the theory of Proof 3.** The bound
$\delta_k \le (1-\alpha\mu)^k\delta_0 + \frac{L\alpha\sigma_B^2}{2\mu}$ predicts a geometric fall onto a
plateau. Therefore:

- A **plateau** means the noise ball has been reached, not that the model has converged. Cutting $\alpha$
  by $10$ drops the floor by $10$ and produces the characteristic staircase in ResNet-style training
  curves.
- A **diverging** or sawtoothing loss means $\alpha \gt 2/L_{\text{local}}$; reduce $\alpha$ or add warmup.
- **Cosine and $1/\sqrt{k}$ schedules** interpolate between the Robbins-Monro requirement (asymptotic
  quiet) and the practical desire to keep $\alpha$ large for most of training.

**Averaging.** Polyak-Juditsky (Theorem 8) says the *average* iterate is statistically optimal even when
the last iterate is noisy. Modern practice implements this as an exponential moving average of weights
(EMA) or stochastic weight averaging (SWA); both routinely gain accuracy at zero training cost.

**Cheap diagnostics.**

| Symptom | Likely cause | Intervention |
|---|---|---|
| Loss plateaus at a nonzero level, weights still moving | noise ball | decay $\alpha$, raise $B$, or average iterates |
| Loss spikes then recovers | $\alpha$ near $2/L$, heavy-tailed gradients | gradient clipping, lower $\alpha$ |
| Loss explodes in the first hundred steps | curvature at initialization | warmup, better initialization |
| Train loss falls, gradient norm stays large | strongly nonconvex or mislabeled data | inspect the noise floor $\sigma^2$ |

### 4.3 Implementing Adaptive Methods Correctly

**AdaGrad's monotone denominator.** $v_{k,j} = \sum_{t\le k}g_{t,j}^2$ never decreases, so the effective
step $\alpha/\sqrt{v_{k,j}}$ decays like $1/\sqrt{k}$ — automatically satisfying a Robbins-Monro-type
schedule, and the reason AdaGrad excels on sparse features (rare coordinates accumulate little, so keep
large steps) but stalls on long deep-learning runs.

**RMSProp and Adam** replace the sum by an exponential moving average, which forgets old curvature and
therefore does *not* decay to zero. That fixes the stalling but forfeits AdaGrad's automatic schedule,
which is why Adam is normally paired with an explicit decay.

**Implementation details that matter.**

- $\varepsilon$ sits **outside** the square root in the standard formulation,
  $\alpha\hat{m}/(\sqrt{\hat{v}}+\varepsilon)$; placing it inside changes the small-gradient behaviour
  materially. Typical $\varepsilon = 10^{-8}$, raised to $10^{-6}$ or higher for mixed-precision training.
- **Weight decay is not $\ell_2$ regularization under Adam.** Adding $\lambda\mathbf{x}$ to the gradient
  makes the penalty pass through the $\sqrt{\hat{\mathbf{v}}}$ denominator and be rescaled per coordinate;
  AdamW instead applies $\mathbf{x} \leftarrow \mathbf{x} - \alpha\lambda\mathbf{x}$ decoupled from the
  adaptive term, which is what the regularization was meant to do.
- **Memory.** Adam stores $\mathbf{m}$ and $\mathbf{v}$, tripling optimizer state relative to plain SGD
  ($2n$ extra floats). At $n = 10^{10}$ parameters this dominates the memory budget and motivates 8-bit
  optimizer states, Adafactor's factored second moments, and sharded (ZeRO) optimizer states.
- **Bias correction can be folded into the step size:** $\alpha_k = \alpha\frac{\sqrt{1-\beta_2^k}}{1-\beta_1^k}$
  applied to the uncorrected moments, saving two vector operations per step.

## 5. Real-World Physics & AI/ML Applications

### 5.1 Training Modern Models

- **Cost accounting.** For a model with $n$ parameters trained on $N$ tokens, each SGD step costs
  $O(nB)$ FLOPs for the forward-backward pass. The Chinchilla-style question "how should a fixed compute
  budget be split between model size and data?" is precisely the budget allocation of Section 4.1, with
  the noise floor determining how much of the budget can usefully be spent on more steps.
- **Why Adam dominates language models.** Gradient scales differ by orders of magnitude across embedding,
  attention and layernorm parameters. Adam's per-coordinate preconditioning (Proof 6) removes that scale
  disparity, effectively reducing the condition number that a single global $\alpha$ must serve.
- **Gradient accumulation** simulates a large $B$ on limited memory by summing micro-batch gradients before
  a single update — mathematically identical to a large batch by Proof 1, with the same $\sigma^2/B$
  variance and the same need for linear-scaling $\alpha$.
- **Label noise sets a floor.** If a fraction $p$ of labels are wrong, $\sigma^2$ is bounded below and no
  step-size schedule can drive the training loss to zero without memorizing the noise — the optimization
  view of the bias-variance trade-off.

### 5.2 Physics: SGD as a Langevin Diffusion

Treat one SGD step as an Euler-Maruyama discretization. With $\mathbf{g}_k = \nabla f + \boldsymbol{\epsilon}_k$
and $\operatorname{Cov}(\boldsymbol{\epsilon}_k) = \frac{\Sigma(\mathbf{x})}{B}$,

$$
\mathbf{x}_{k+1} = \mathbf{x}_k - \alpha\nabla f(\mathbf{x}_k) - \alpha\boldsymbol{\epsilon}_k \ \approx\ d\mathbf{x} = -\nabla f(\mathbf{x})\,dt + \sqrt{\frac{\alpha}{B}\,\Sigma(\mathbf{x})}\ d\mathbf{W}_t
$$

with $dt = \alpha$. For isotropic $\Sigma = \sigma^2I$ this is **overdamped Langevin dynamics** at
temperature

$$
T = \frac{\alpha\sigma^2}{B}
$$

whose stationary distribution is the Gibbs measure $p(\mathbf{x}) \propto \exp\left(-\frac{2f(\mathbf{x})}{T}\right)$.
Three consequences follow directly:

- **The temperature is the single knob.** Only the ratio $\alpha/B$ matters, which is exactly the linear
  scaling rule of Section 4.1, now derived from physics rather than from a variance bound.
- **Escape from saddles and shallow minima.** Kramers' escape-rate law gives expected escape time
  $\tau \sim \exp(2\Delta E/T)$ over a barrier of height $\Delta E$: high temperature (large $\alpha/B$)
  escapes quickly, low temperature settles. Annealing $\alpha$ is literally simulated annealing.
- **Implicit regularization toward flat minima.** The Gibbs measure weights a basin by its *volume*,
  $\int_{\text{basin}}e^{-2f/T}$, which for a quadratic basin with Hessian $H$ scales like
  $\left(\det H\right)^{-1/2}$. Flat (small $\det H$) minima therefore attract exponentially more
  probability mass, and generalization correlates with flatness. Gradient noise is a *feature*.

**Anisotropy matters.** In deep networks $\Sigma(\mathbf{x}) \approx \nabla^2 f(\mathbf{x})$ near a minimum
(the Fisher/Gauss-Newton approximation), so the noise is *aligned with curvature*: SGD injects the most
noise exactly along the sharpest directions, which strengthens the flat-minimum bias beyond what isotropic
Langevin predicts.

### 5.3 Case Study: SGD versus Adam on an Ill-Conditioned Quadratic

Take $f(\mathbf{x}) = \frac12\mathbf{x}^TA\mathbf{x}$ with $A = \operatorname{diag}(1, 100)$
($\kappa = 100$) and isotropic gradient noise of variance $\sigma^2$ per coordinate.

**Plain SGD.** Stability requires $\alpha \lt 2/L = 0.02$, so the slow coordinate contracts by at most
$1 - \alpha\mu \approx 0.99$ per step: roughly $230$ steps per decimal digit, and a per-coordinate noise
floor

$$
\mathbb{E}\left[x_j^2\right]_\infty \approx \frac{\alpha\sigma^2}{2\lambda_j - \alpha\lambda_j^2} \approx \frac{\alpha\sigma^2}{2\lambda_j}
$$

so the flat coordinate ($\lambda = 1$) carries $100\times$ more residual variance than the stiff one.

**Adam.** The denominator $\sqrt{\hat{v}_j}$ estimates the root-mean-square gradient in coordinate $j$,
which is $\approx\lambda_j\lvert x_j\rvert$ in the signal-dominated regime. The effective per-coordinate
step is then

$$
\alpha_{\mathrm{eff},j} \approx \frac{\alpha}{\lambda_j\lvert x_j\rvert}\cdot\lambda_j\lvert x_j\rvert \ \text{-normalized} \implies \text{updates of size} \approx \alpha \ \text{in every coordinate}
$$

so both coordinates make comparable progress: the effective condition number is $\approx 1$ and the
transient is $\kappa$-independent. In the noise-dominated regime, however, $\sqrt{\hat{v}_j} \to \sigma$
for all $j$ and the preconditioner becomes a uniform $1/\sigma$: Adam's advantage is a *transient* one,
which is why Adam runs still need a decay schedule to reach high accuracy, and why well-tuned SGD with
momentum can match Adam on well-conditioned vision tasks while losing badly on transformers.

The companion exercises quantify all of these statements on explicit numbers.

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source | Location |
|---|---|---|
| Stochastic approximation, step-size conditions (Definition 5, Proof 4) | Robbins & Monro, *Ann. Math. Statist.* 22 (1951) | Full paper |
| Unbiasedness, noise ball, batching, complexity (Proofs 1-3, Theorem 5) | Bottou, Curtis & Nocedal, *SIAM Review* 60 (2018) | Sections 4-5 |
| Minimax $\Omega(1/k)$ lower bound for strongly convex stochastic optimization | Nemirovski & Yudin, *Problem Complexity and Method Efficiency* | Chapters 5-6 |
| Smoothness, strong convexity, accelerated rates targeted by stochastic methods | Nesterov, *Introductory Lectures on Convex Optimization* | Chapter 2 |
| Iterate averaging and asymptotic optimality (Theorem 8) | Polyak & Juditsky, *SIAM J. Control Optim.* 30 (1992) | Full paper |
| SVRG control variate and linear rate (Proof 5, Theorem 6) | Johnson & Zhang, *NeurIPS* (2013) | Sections 2-3 |
| AdaGrad and per-coordinate scaling (Definition 8) | Duchi, Hazan & Singer, *JMLR* 12 (2011) | Sections 1-3 |
| Adam and the bias-correction derivation (Proof 6B, Theorem 7) | Kingma & Ba, *ICLR* (2015) | Sections 2-3 |
| Decoupled weight decay (Section 4.3) | Loshchilov & Hutter, *ICLR* (2019) | Algorithm 2 |
| Linear scaling rule, warmup, large-batch training (Section 4.1) | Goyal et al. (2017); McCandlish et al. (2018) | Sections 2-3 |
| SGD as Langevin diffusion, flat minima (Section 5.2) | Mandt, Hoffman & Blei, *JMLR* 18 (2017); Jastrzebski et al. (2018) | Full papers |
| Saddle escape with noise | Ge, Huang, Jin & Yuan, *COLT* (2015); Jin et al., *ICML* (2017) | Main theorems |
| Practical optimization for deep models | Goodfellow, Bengio & Courville, *Deep Learning* | Chapter 8 |

**Suggested reading order.** Start with Bottou-Curtis-Nocedal Sections 4-5 for the unified treatment
reproduced in Proofs 1-4; then Robbins-Monro for the original schedule argument; then Johnson-Zhang for
variance reduction and Kingma-Ba for adaptive methods; finish with Mandt-Hoffman-Blei for the
diffusion picture that makes the temperature $\alpha\sigma^2/B$ explicit. The exercises notebook in this
folder turns every theorem above into arithmetic on concrete batch sizes, learning rates and noise floors.